# Why perfect fold AUC can coexist with weaker global OOF AUC

**Data Scientist / Computer Vision portfolio — Décimus Tavares**

This notebook uses fabricated arrays only. It illustrates an evaluation issue
encountered during RSNA knee MRI experimentation. It neither reproduces MRI
predictions nor recalibrates original OOF probabilities.

Install the package from the repository root with `python -m pip install -e .`.


In [1]:
from knee_portfolio.evaluation import pair_decomposition
import json

# Two folds; in each fold the positive scores above its negative.
result = pair_decomposition(
    target=[0, 1, 0, 1],
    prediction=[0.1, 0.2, 0.8, 0.9],
    folds=[1, 1, 2, 2],
)
print(json.dumps(result, indent=2))
assert result['mean_defined_fold_auc'] == 1.0
assert result['global']['auc'] == 0.75


{
  "known": 4,
  "unknown": 0,
  "global": {
    "pairs": 4,
    "concordance_sum": 3.0,
    "auc": 0.75
  },
  "within": {
    "pairs": 2,
    "concordance_sum": 2.0,
    "auc": 1.0
  },
  "between": {
    "pairs": 2,
    "concordance_sum": 1.0,
    "auc": 0.5
  },
  "per_fold": [
    {
      "pairs": 1,
      "concordance_sum": 1.0,
      "auc": 1.0,
      "fold": "1",
      "positives": 1,
      "negatives": 1
    },
    {
      "pairs": 1,
      "concordance_sum": 1.0,
      "auc": 1.0,
      "fold": "2",
      "positives": 1,
      "negatives": 1
    }
  ],
  "mean_defined_fold_auc": 1.0
}


## Interpretation

The positive in fold 1 scores below the negative in fold 2. That comparison never
appears in a fold-local AUC, but it counts in global AUC. The global value weights
all positive-negative pairs; the simple fold mean weights defined folds equally.
Score scales and sample composition can both matter. This example does not establish
which mechanism dominated a real model, and no score transformation is applied.


In [2]:
from knee_portfolio.demo import main

# Also validates exact merge and global ranking with synthetic study names.
main()


{
  "data": "synthetic only",
  "auc_decomposition": {
    "known": 4,
    "unknown": 0,
    "global": {
      "pairs": 4,
      "concordance_sum": 3.0,
      "auc": 0.75
    },
    "within": {
      "pairs": 2,
      "concordance_sum": 2.0,
      "auc": 1.0
    },
    "between": {
      "pairs": 2,
      "concordance_sum": 1.0,
      "auc": 0.5
    },
    "per_fold": [
      {
        "pairs": 1,
        "concordance_sum": 1.0,
        "auc": 1.0,
        "fold": "1",
        "positives": 1,
        "negatives": 1
      },
      {
        "pairs": 1,
        "concordance_sum": 1.0,
        "auc": 1.0,
        "fold": "2",
        "positives": 1,
        "negatives": 1
      }
    ],
    "mean_defined_fold_auc": 1.0
  },
  "shard_merge_and_global_rank_equal": true
}


## Project evidence and limits

The project-developed EXP04 achieved audited OOF 0.632989 and a participant-reported
accepted public score of 0.732. An attributed Raptor reproduction later achieved
public 0.924. These are different evaluation settings and different model provenance.

EXP12 tested two-GPU inference without changing the model recipe, but missed the
official submission deadline. Neither this notebook nor the visible-three test
establishes hidden-test completion, final rank or clinical effectiveness.

See `docs/experiments.md` for retained negative results and `docs/provenance.md`
for original model attribution.
